# Image Classifier with Transfer Learning

Train a **pretrained ResNet-18** on a Fashion-MNIST class subset using PyTorch.

**Pipeline:** data augmentation → transfer learning → training curves → evaluation metrics → saved model.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Config

In [ ]:
SEED = 42
EPOCHS = 3
BATCH_SIZE = 32
LR = 1e-3
MAX_TRAIN_SAMPLES = 4000  # set 0 to use all
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
CLASS_NAMES = ["Trouser", "Dress", "Sandal", "Sneaker", "Bag"]
CLASS_INDICES = [ALL_CLASSES.index(c) for c in CLASS_NAMES]
NUM_CLASSES = len(CLASS_NAMES)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Classes:", CLASS_NAMES)

## 2. Data & augmentation

Grayscale → RGB for ResNet. Training uses **random crop**, **horizontal flip**, and **color jitter**.

In [ ]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(224),
    transforms.RandomCrop(224, padding=16),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


def filter_by_classes(dataset, class_indices):
    indices = [i for i, y in enumerate(dataset.targets) if int(y) in class_indices]
    remap = {old: new for new, old in enumerate(class_indices)}

    class RemappedSubset(Subset):
        def __getitem__(self, idx):
            img, label = super().__getitem__(idx)
            return img, remap[int(label)]

        def __getitems__(self, indices):
            return [self.__getitem__(idx) for idx in indices]

    return RemappedSubset(dataset, indices)


train_full = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=train_tf)
test_full = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=eval_tf)
train_ds = filter_by_classes(train_full, CLASS_INDICES)
test_ds = filter_by_classes(test_full, CLASS_INDICES)

indices = list(range(len(train_ds)))
random.shuffle(indices)
if MAX_TRAIN_SAMPLES > 0:
    indices = indices[:MAX_TRAIN_SAMPLES]
val_size = max(1, int(0.1 * len(indices)))
train_subset = Subset(train_ds, indices[val_size:])
val_subset = Subset(train_ds, indices[:val_size])

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train={len(train_subset)} Val={len(val_subset)} Test={len(test_ds)}")

## 3. Model (pretrained ResNet-18)

In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, NUM_CLASSES),
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
print(model.fc)

## 4. Train

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(1).cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
    y_true, y_pred = np.array(all_labels), np.array(all_preds)
    return running_loss / len(y_true), accuracy_score(y_true, y_pred), y_true, y_pred


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
best_path = OUTPUT_DIR / "best_model.pt"

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    print(f"  train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": CLASS_NAMES,
            "num_classes": NUM_CLASSES,
            "dataset": "fashionmnist",
            "epoch": epoch,
            "val_acc": val_acc,
            "freeze_backbone": True,
        }, best_path)
        print("  saved best model")

## 5. Training curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], marker="o", label="Train")
axes[0].plot(epochs, history["val_loss"], marker="s", label="Val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history["train_acc"], marker="o", label="Train")
axes[1].plot(epochs, history["val_acc"], marker="s", label="Val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Test evaluation

In [ ]:
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])

test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)
macro_f1 = f1_score(y_true, y_pred, average="macro")
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)

print(f"Test accuracy: {test_acc:.4f}")
print(f"Macro F1:      {macro_f1:.4f}")
print(report)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "dataset": "fashionmnist",
        "test_accuracy": float(test_acc),
        "macro_f1": float(macro_f1),
        "best_val_accuracy": float(best_val_acc),
        "class_names": CLASS_NAMES,
        "history": history,
        "classification_report": report,
    }, f, indent=2)

print("Saved:", best_path)

## 7. Inference

```bash
python inference.py --image path/to/image.jpg --model outputs/best_model.pt
```